# 01 - Bronze Ingestion (Auto Loader)
Incrementally loads raw CSV files from the Unity Catalog Volume into Bronze Delta tables.
- Auto Loader (`cloudFiles`) only processes **new** files, tracked by checkpoints
- Data is kept as-is (all strings), with metadata columns added for lineage
- `trigger(availableNow=True)` processes everything new, then stops (works on serverless)

In [0]:
from pyspark.sql import functions as F

catalog = "retail_lakehouse"
raw_base = f"/Volumes/{catalog}/bronze/raw_files"
checkpoint_base = f"/Volumes/{catalog}/bronze/checkpoints"

# source folder -> bronze table name
sources = {
    "orders":      "orders_raw",
    "order_items": "order_items_raw",
    "customers":   "customers_raw",
    "products":    "products_raw",
    "payments":    "payments_raw",
}

In [0]:
def ingest_to_bronze(source_folder: str, table_name: str):
    """Use Auto Loader to stream new CSV files from a volume folder into a Bronze Delta table."""
    target_table = f"{catalog}.bronze.{table_name}"

    query = (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("header", "true")
            .option("cloudFiles.schemaLocation", f"{checkpoint_base}/{source_folder}/_schema")
            .load(f"{raw_base}/{source_folder}")
            .select(
                "*",
                F.current_timestamp().alias("_ingested_at"),
                F.col("_metadata.file_path").alias("_source_file"),
            )
        .writeStream
            .option("checkpointLocation", f"{checkpoint_base}/{source_folder}/_checkpoint")
            .trigger(availableNow=True)
            .toTable(target_table)
    )

    query.awaitTermination()
    print(f"✅ {source_folder:12} -> {target_table}")

In [0]:
for source_folder, table_name in sources.items():
    ingest_to_bronze(source_folder, table_name)

In [0]:
%sql
SELECT 'orders_raw'      AS table_name, COUNT(*) AS row_count FROM retail_lakehouse.bronze.orders_raw
UNION ALL SELECT 'order_items_raw', COUNT(*) FROM retail_lakehouse.bronze.order_items_raw
UNION ALL SELECT 'customers_raw',   COUNT(*) FROM retail_lakehouse.bronze.customers_raw
UNION ALL SELECT 'products_raw',    COUNT(*) FROM retail_lakehouse.bronze.products_raw
UNION ALL SELECT 'payments_raw',    COUNT(*) FROM retail_lakehouse.bronze.payments_raw;

In [0]:
%sql
SELECT order_id, order_status, _ingested_at, _source_file, _rescued_data
FROM retail_lakehouse.bronze.orders_raw
LIMIT 5;

In [0]:
%sql
COMMENT ON TABLE retail_lakehouse.bronze.orders_raw      IS 'Raw orders from Olist CSVs, loaded by Auto Loader. No transformations.';
COMMENT ON TABLE retail_lakehouse.bronze.order_items_raw IS 'Raw order line items from Olist CSVs, loaded by Auto Loader.';
COMMENT ON TABLE retail_lakehouse.bronze.customers_raw   IS 'Raw customers from Olist CSVs, loaded by Auto Loader.';
COMMENT ON TABLE retail_lakehouse.bronze.products_raw    IS 'Raw products from Olist CSVs, loaded by Auto Loader.';
COMMENT ON TABLE retail_lakehouse.bronze.payments_raw    IS 'Raw payments from Olist CSVs, loaded by Auto Loader.';

In [0]:
%sql
DESCRIBE HISTORY retail_lakehouse.bronze.orders_raw;

In [0]:
%sql
SELECT _source_file, COUNT(*) AS rows_loaded, MAX(_ingested_at) AS loaded_at
FROM retail_lakehouse.bronze.orders_raw
GROUP BY _source_file
ORDER BY loaded_at;